## Alternative metrics on cellxgene counts, batch dataset, kl1

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import sys

sys.path.append("../../existing_metric_comparison/")
sys.path.append("../../utils/")
import data_cleanup
import metrics_lib as metrics
import os

## Set global variables

In [ ]:
study = "Sheppard_2020"
batch_variable = "dataset"
kl = "1.0"
use_old_counts = False
if use_old_counts == True:
    suffix = "original_counts_"
else:
    suffix = "cellxgene_counts_"

path_emb = f"/lustre/groups/ml01/workspace/lisa.sikkema/atlas_extension/data/HLCA/HLCA_and_{study}_{suffix}batch_{batch_variable}_kl{kl}_latent.h5ad"
path_counts = f"/lustre/groups/ml01/workspace/lisa.sikkema/atlas_extension/data/HLCA/HLCA_and_{study}_counts.h5ad"
path_results_dir = f"/lustre/groups/ml01/code/lisa.sikkema/atlas_extension/results/HLCA/{study}/{suffix}batch_{batch_variable}_kl{kl}/mapqc"

In [ ]:
RESULT_DIR = (
    f"../../../results/HLCA/{study}/{suffix}batch_{batch_variable}_kl{kl}/benchmark"
)

In [ ]:
if not os.path.exists(RESULT_DIR):
    if os.path.exists(os.path.dirname(RESULT_DIR)):
        print(f"Creating directory {RESULT_DIR}")
        os.mkdir(RESULT_DIR)
    else:
        print("Parent directory of RESULT_DIR does not exist. Create it first.")

## load and prep Data

In [ ]:
adata_emb = sc.read_h5ad(path_emb)
adata_counts = sc.read_h5ad(path_counts)
adata_full = data_cleanup.add_mapping_info_to_counts_adata(adata_counts, adata_emb)

## run metrics

### Query healthy

subset to condition of interest? 

In [ ]:
query_condition_1 = "Healthy"

In [ ]:
adata_1 = adata_full[
    (adata_full.obs.lung_condition == query_condition_1)
    | (adata_full.obs["r_or_q"] == "r")
].copy()

subset query for metric functions below

In [ ]:
adata_query_1 = adata_1[adata_1.obs.study == "Sheppard_2020", :].copy()

set some variables for running metrics

In [ ]:
obsm_emb_key = "X_scanvi_emb"
donor_key = "donor_id"
dataset_key = "r_or_q"
sample_key = "sample"
original_emb_key = None
batch_key = "r_or_q"

get results for other metrics:

In [ ]:
scib_results_1 = metrics.calculate_scib_metrics(
    adata_1, obsm_emb_key, batch_key, dataset_key, sample_key, dry_run=False
)

In [ ]:
symphony_results_1 = metrics.calculate_symphony_metrics(
    adata_1,
    adata_query_1,
    obsm_emb_key,
    donor_key,
    dataset_key,
    sample_key,
    dry_run=False,
)

In [ ]:
other_results_1 = metrics.calculate_other_metrics(
    adata_1, adata_query_1, obsm_emb_key, dataset_key, sample_key, dry_run=False
)

In [ ]:
# save results
adata_1.obs["lisi_batch"] = scib_results_1["lisi_batch"]["per_cell"]
adata_1.obs["kbet_batch"] = scib_results_1["kbet_batch"]["per_cell"]
adata_1.obs["lisi_query_donor"] = symphony_results_1["lisi_query_donor"]["per_cell"]
adata_1.obs["wiq_knn_corr"] = symphony_results_1["wiq_knn_corr"]["per_cell"]
adata_1.obs["cluster_preservation_score"] = other_results_1[
    "cluster_preservation_score"
]["per_cell"]

In [ ]:
adata_1.obs[
    [
        "lisi_batch",
        "kbet_batch",
        "lisi_query_donor",
        "wiq_knn_corr",
        "cluster_preservation_score",
    ]
].to_csv(os.path.join(RESULT_DIR, f"alt_metrics_{query_condition_1.lower()}.csv"))

## Visualize 

Run Load and prep embeddings and set global variables first

In [ ]:
adata_1.obs[
    [
        "lisi_batch",
        "kbet_batch",
        "lisi_query_donor",
        "wiq_knn_corr",
        "cluster_preservation_score",
    ]
] = pd.read_csv(
    os.path.join(RESULT_DIR, f"alt_metrics_{query_condition_1.lower()}.csv"),
    index_col="Unnamed: 0",
)

In [ ]:
for metric in [
    "lisi_batch",
    "kbet_batch",
    "lisi_query_donor",
    "wiq_knn_corr",
    "cluster_preservation_score",
]:
    adata_1.obs.loc[adata_1.obs.core_or_extension == "core", metric] = np.nan
    sc.pl.umap(adata_1, color=metric)  # ,vmax='p99')

### Query disease

In [ ]:
query_condition_2 = "IPF"

In [ ]:
adata_2 = adata_full[
    (adata_full.obs.lung_condition == query_condition_2)
    | (adata_full.obs["r_or_q"] == "r")
].copy()

In [ ]:
del adata_full

subset query for metric functions below

In [ ]:
adata_query_2 = adata_2[adata_2.obs.study == "Sheppard_2020", :].copy()

set some variables for running metrics

In [ ]:
obsm_emb_key = "X_scanvi_emb"
donor_key = "donor_id"
dataset_key = "r_or_q"
sample_key = "sample"
original_emb_key = None
batch_key = "r_or_q"

get results for other metrics:

In [ ]:
scib_results_2 = metrics.calculate_scib_metrics(
    adata_2, obsm_emb_key, batch_key, dataset_key, sample_key, dry_run=False
)

In [ ]:
symphony_results_2 = metrics.calculate_symphony_metrics(
    adata_2,
    adata_query_2,
    obsm_emb_key,
    donor_key,
    dataset_key,
    sample_key,
    dry_run=False,
)

In [ ]:
other_results_2 = metrics.calculate_other_metrics(
    adata_2, adata_query_2, obsm_emb_key, dataset_key, sample_key, dry_run=False
)

In [ ]:
# save results
adata_2.obs["lisi_batch"] = scib_results_2["lisi_batch"]["per_cell"]
adata_2.obs["kbet_batch"] = scib_results_2["kbet_batch"]["per_cell"]
adata_2.obs["lisi_query_donor"] = symphony_results_2["lisi_query_donor"]["per_cell"]
adata_2.obs["wiq_knn_corr"] = symphony_results_2["wiq_knn_corr"]["per_cell"]
adata_2.obs["cluster_preservation_score"] = other_results_2[
    "cluster_preservation_score"
]["per_cell"]

In [ ]:
adata_2.obs[
    [
        "lisi_batch",
        "kbet_batch",
        "lisi_query_donor",
        "wiq_knn_corr",
        "cluster_preservation_score",
    ]
].to_csv(os.path.join(RESULT_DIR, f"alt_metrics_{query_condition_2.lower()}.csv"))

## Visualize 

Run Load and prep embeddings and set global variables first

In [ ]:
adata_2.obs[
    [
        "lisi_batch",
        "kbet_batch",
        "lisi_query_donor",
        "wiq_knn_corr",
        "cluster_preservation_score",
    ]
] = pd.read_csv(
    os.path.join(RESULT_DIR, f"alt_metrics_{query_condition_2.lower()}.csv"),
    index_col="Unnamed: 0",
)

In [ ]:
for metric in [
    "lisi_batch",
    "kbet_batch",
    "lisi_query_donor",
    "wiq_knn_corr",
    "cluster_preservation_score",
]:
    adata_2.obs.loc[adata_2.obs.core_or_extension == "core", metric] = np.nan
    sc.pl.umap(adata_2, color=metric)  # ,vmax='p99')

In [ ]:
print("test")